# AME 5052 – Applied Machine Learning Lab
## Assignment 3: Performance Analysis and Comparison of Models

**Question:** Analyse the performance of the different models developed in Questions 1 and 2 using appropriate metrics.

This notebook compares the regression and classification models from Assignment 1 and clustering and ensemble models from Assignment 2. It uses consistent train/test splits and evaluation metrics. Dataset choices are included so the notebook can be run independently.

## 3(a). Regression model comparison (Assignment 1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             silhouette_score, adjusted_rand_score)

# Regression data
diabetes = load_diabetes(as_frame=True)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42
)

reg_models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10000))
}
reg_rows = []
for name, model in reg_models.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    reg_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(yr_test, pred),
        "MSE": mean_squared_error(yr_test, pred),
        "RMSE": np.sqrt(mean_squared_error(yr_test, pred)),
        "R2": r2_score(yr_test, pred)
    })

reg_comparison = pd.DataFrame(reg_rows).sort_values("RMSE")
display(reg_comparison.round(4))

**Interpretation:** For regression, MAE, MSE and RMSE measure prediction errors, so lower values are preferred. R² measures explained variance, so a higher value is preferred. The table provides a direct comparison on the same test set.

## 3(b). Classification model comparison (Assignment 1 and ensemble models from Assignment 2)

In [ ]:
# Use Iris for a consistent multi-class comparison of the classification algorithms
iris = load_iris(as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

classification_models = {
    "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

class_rows = []
for name, model in classification_models.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    class_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(yc_test, pred),
        "Precision (macro)": precision_score(yc_test, pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(yc_test, pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(yc_test, pred, average="macro", zero_division=0)
    })

class_comparison = pd.DataFrame(class_rows).sort_values("F1 (macro)", ascending=False)
display(class_comparison.round(4))

In [ ]:
class_comparison.set_index("Model")[["Accuracy", "F1 (macro)"]].plot(
    kind="bar", figsize=(11, 5), ylim=(0, 1.05)
)
plt.ylabel("Score")
plt.title("Classification Model Performance Comparison")
plt.xticks(rotation=35, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

**Interpretation:** Accuracy measures the proportion of correct predictions. Precision measures how many predicted positives are correct; recall measures how many actual positives are identified. F1-score is the harmonic mean of precision and recall. Macro averaging gives equal weight to each class. A model with a higher score is not automatically best for every application; the cost of false positives and false negatives should also be considered.

## 3(c). Clustering model comparison (Assignment 2)

In [ ]:
# Standardize features before distance-based clustering
X_cluster = StandardScaler().fit_transform(iris.data)
cluster_models = {
    "K-Means": KMeans(n_clusters=3, random_state=42, n_init=10),
    "Hierarchical (Ward)": AgglomerativeClustering(n_clusters=3, linkage="ward"),
    "GMM": GaussianMixture(n_components=3, random_state=42)
}

cluster_rows = []
for name, model in cluster_models.items():
    labels = model.fit_predict(X_cluster)
    cluster_rows.append({
        "Model": name,
        "Silhouette Score": silhouette_score(X_cluster, labels),
        "Adjusted Rand Index": adjusted_rand_score(iris.target, labels)
    })

cluster_comparison = pd.DataFrame(cluster_rows).sort_values("Silhouette Score", ascending=False)
display(cluster_comparison.round(4))

**Interpretation:** Silhouette score ranges from -1 to 1; a higher value generally indicates better-defined and better-separated clusters. Adjusted Rand index compares cluster assignments with known class labels when such labels are available; 1 indicates perfect agreement, while values near 0 indicate chance-level agreement. In real unsupervised problems, true labels may not be available, so internal metrics such as silhouette score are especially useful.

## 3(d). PCA analysis

In [ ]:
pca = PCA(n_components=2)
X_two_components = pca.fit_transform(X_cluster)
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative explained variance:", round(pca.explained_variance_ratio_.sum(), 4))

plt.figure(figsize=(8, 5))
plt.scatter(X_two_components[:, 0], X_two_components[:, 1],
            c=iris.target, cmap="viridis", s=35)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Projection of Iris Dataset")
plt.grid(True)
plt.show()

## Overall conclusion

The models are evaluated with metrics appropriate to their learning tasks:

- **Regression:** MAE, MSE, RMSE and R².
- **Classification:** Accuracy, precision, recall and F1-score.
- **Clustering:** Silhouette score and adjusted Rand index (the latter uses known labels for external validation).
- **PCA:** Explained variance ratio.

The comparison tables should be used to identify the strongest model for each task. Regression, classification and clustering metrics are not directly comparable with one another because they measure different objectives. PCA is a dimensionality-reduction technique rather than a predictive model, so explained variance is used to assess how much information the selected components retain.